<a href="https://colab.research.google.com/github/montaha335/ExamTPF_montaha_benmed/blob/main/TP04_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opencv-python numpy matplotlib

In [2]:

!wget -O att_faces.zip https://www.cl.cam.ac.uk/research/dtg/attarchive/pub/data/att_faces.zip
!unzip -o -q att_faces.zip
!mkdir -p att_faces
!mv s* att_faces/
import os
print(os.listdir("att_faces"))

--2026-03-11 03:56:15--  https://www.cl.cam.ac.uk/research/dtg/attarchive/pub/data/att_faces.zip
Resolving www.cl.cam.ac.uk (www.cl.cam.ac.uk)... 128.232.0.20, 2a05:b400:110::80:14
Connecting to www.cl.cam.ac.uk (www.cl.cam.ac.uk)|128.232.0.20|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3769022 (3.6M) [application/zip]
Saving to: ‘att_faces.zip’

att_faces.zip       100%[===================>]   3.59M  3.47MB/s    in 1.0s    

2026-03-11 03:56:17 (3.47 MB/s) - ‘att_faces.zip’ saved [3769022/3769022]

['s3', 's34', 's21', 's5', 's36', 's7', 's33', 's8', 's1', 's17', 'sample_data', 's29', 's32', 's9', 's18', 's27', 's26', 's39', 's24', 's16', 's12', 's2', 's4', 's40', 's19', 's20', 's14', 's30', 's10', 's31', 's22', 's6', 's11', 's38', 's13', 's28', 's25', 's35', 's23', 's15', 's37']


In [3]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

In [11]:
import cv2
import numpy as np
import os

class FaceRecognitionPCA:
    def __init__(self, k=20):
        self.k = k
        # Correcting the typo: 'haascades' -> 'haarcascades' and 'haascade' -> 'haarcascade'
        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )


    def detect_face(self, image):
        if image is None:
            return None

        # Check if the image is already grayscale (1 channel) or convert if it's BGR
        if len(image.shape) == 2 or image.shape[2] == 1:
            gray = image
        else:
            # If it's a color image (3 channels), convert to grayscale
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        faces = self.face_cascade.detectMultiScale(gray, 1.3, 5)
        if len(faces) == 0:
            return None
        (x, y, w, h) = faces[0]
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (100, 100))
        return face


    def prepare_training_set(self, folder):
        data = []
        labels = []
        for person in os.listdir(folder):
            person_path = os.path.join(folder, person)
            if os.path.isdir(person_path):
                for img in os.listdir(person_path):
                    path = os.path.join(person_path, img)
                    image = cv2.imread(path, cv2.IMREAD_UNCHANGED)
                    if image is None:
                        print(f"Image non lue : {path}")
                        continue
                    face = self.detect_face(image)
                    if face is not None:
                        data.append(face.flatten())
                        labels.append(person)
        self.X = np.array(data)
        self.labels = labels


    def compute_pca(self):
        self.mean = np.mean(self.X, axis=0)
        A = self.X - self.mean
        cov = np.cov(A, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvectors = eigenvectors[:, idx]
        self.eigenfaces = eigenvectors[:, :self.k]


    def project(self, face):
        face = face.flatten()
        centered = face - self.mean
        return np.dot(centered, self.eigenfaces)


    def recognize(self, image, threshold=3000):
        face = self.detect_face(image)
        if face is None:
            return None, None, "No face detected"
        proj_test = self.project(face)
        min_dist = float('inf')
        identity = None
        for i, x in enumerate(self.X):
            proj_train = self.project(x.reshape(100,100))
            dist = np.linalg.norm(proj_test - proj_train)
            if dist < min_dist:
                min_dist = dist
                identity = self.labels[i]
        decision = "Match" if min_dist < threshold else "No Match"
        return identity, min_dist, decision

In [13]:
model = FaceRecognitionPCA(k=20)
model.prepare_training_set("att_faces")
model.compute_pca()

Image non lue : att_faces/sample_data/mnist_test.csv
Image non lue : att_faces/sample_data/california_housing_test.csv
Image non lue : att_faces/sample_data/california_housing_train.csv
Image non lue : att_faces/sample_data/mnist_train_small.csv
Image non lue : att_faces/sample_data/README.md
Image non lue : att_faces/sample_data/anscombe.json


In [14]:
print("Nombre d'images :", model.X.shape[0])
print("Taille d'une image aplatie :", model.X.shape[1])
print("Nombre d'éigenfaces calculées :", model.eigenfaces.shape[1])

Nombre d'images : 263
Taille d'une image aplatie : 10000
Nombre d'éigenfaces calculées : 20


In [15]:
os.listdir("att_faces/s1")

['6.pgm',
 '8.pgm',
 '10.pgm',
 '3.pgm',
 '7.pgm',
 '4.pgm',
 '5.pgm',
 '2.pgm',
 '1.pgm',
 '9.pgm']

In [18]:
!cp att_faces/s1/1.pgm test.pgm

In [19]:
img = cv2.imread("test.pgm", cv2.IMREAD_UNCHANGED)
identity, dist, decision = model.recognize(img)
print("Identité :", identity)
print("Distance :", dist)
print("Decision :", decision)

Identité : s1
Distance : 0.0
Decision : Match
